In [1]:
from ingest import load_faq_data,build_index
from tqdm.auto import tqdm
import numpy as np
from minsearch import VectorSearch
from openai import OpenAI
from rag_helper import RAGBase
from dotenv import load_dotenv
import os

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
documents = load_faq_data()
index = build_index(documents)

In [4]:
load_dotenv()
openai_client = OpenAI(
     api_key=os.getenv("GROQ_API_KEY"),
     base_url="https://api.groq.com/openai/v1"
)

In [5]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    model="openai/gpt-oss-120b"
)

In [6]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes—you can still join the program. If you’d like to earn a certificate, just be sure to submit your capstone project before the submission deadline ends.'

In [7]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [8]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [9]:
X = np.array(vectors)

In [10]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [11]:
scores = X.dot(v_query)

In [12]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [13]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294106))

In [14]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [15]:
top5 = np.argsort(scores)[-5:]

In [16]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [17]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294106
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579373
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Relate

In [18]:
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [19]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [20]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [21]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [22]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [23]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [24]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes—you can still join even though the program has already started.  Just keep in mind that if you want to earn a certificate you’ll need to submit your capstone project while the submission window is still open.'